```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b,A2,A3,A4a,A4b,A5a,A5b,A6a,A6b,A7,A8a,A8b,A9 done;
    class A10 current;
    class A11,A12 normal;
```

# Notebook 10 — Custom NER: Concept Annotation and Lightweight Training

This notebook introduces custom Named Entity Recognition (NER) for **philosophical concepts** using a lightweight CPU-friendly workflow. Students annotate philosophical concepts using the **BIO tagging scheme**, validate annotations, compute simple inter-annotator agreement, convert annotations into spaCy training data, and train a lightweight NER model suitable for laptops.

The broader methodological goal is to investigate how philosophical concepts can be operationalized computationally and what kinds of limitations emerge when abstract concepts are treated as annotation targets.

## Learning goals

By the end of this notebook, students should be able to:

- understand BIO tagging for sequence labeling
- annotate philosophical concepts consistently
- discuss annotation ambiguity and span boundaries
- validate annotation quality
- compute simple inter-annotator agreement
- convert BIO annotations into spaCy training examples
- train a lightweight custom NER model on CPU
- evaluate model outputs critically
- perform qualitative error analysis
- reflect on the limits of concept extraction in philosophy

## Method note

In this course we annotate **one custom entity label**:

- `CONCEPT`

This is a deliberate simplification. Philosophical corpora contain many possible annotation targets (people, works, schools, concepts, methods, doctrines), but concept annotation is already difficult enough because concepts are abstract, historically unstable, and often realized in multi-word expressions.

For this reason, the notebook emphasizes:

- **clear span policy**
- **annotation consistency**
- **validation before training**
- **evaluation-by-inspection** rather than metric worship
- **responsible reporting** of limitations

## Annotation guidelines

We annotate only one custom entity label:

- `CONCEPT`

Examples:
- practical reason
- virtue
- natural law
- free will
- categorical imperative

### Span policy
Annotate the **smallest semantically meaningful span**.

Preferred:
- `practical reason`

Avoid:
- `the faculty of practical reason`

### BIO tagging
BIO labels indicate:
- `B-CONCEPT`: beginning of a concept span
- `I-CONCEPT`: continuation of a concept span
- `O`: outside any concept span

### Reflection question
Which concepts are easy to annotate consistently? Which concepts become difficult because they are context-dependent, historically unstable, or nested inside larger phrases?

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re
from collections import Counter
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

import spacy
from spacy.tokens import DocBin
from spacy.training import Example
from spacy.util import minibatch

from sklearn.metrics import classification_report

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
PROJECT_ROOT = Path('.')

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'

ANALYSIS_DIR = PROJECT_ROOT / 'analysis'
FIGURES_DIR = ANALYSIS_DIR / 'figures'
TABLES_DIR = ANALYSIS_DIR / 'tables'
REPORTS_DIR = ANALYSIS_DIR / 'reports'
MODELS_DIR = ANALYSIS_DIR / 'models'
CACHE_DIR = PROJECT_ROOT / 'cache'

for p in [FIGURES_DIR, TABLES_DIR, REPORTS_DIR, MODELS_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DOC_INDEX = TABLES_DIR / 'nb03-doc_index.csv'
SPLIT_DIR = PROCESSED_DIR / 'nb05-corpus-split'
ANNOTATION_DIR = PROCESSED_DIR / 'annotations'
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = CACHE_DIR / 'nb10-train_examples.spacy'
DEV_PATH = CACHE_DIR / 'nb10-dev_examples.spacy'
MODEL_DIR = MODELS_DIR / 'nb10-concept_ner'

print('DOC_INDEX:', DOC_INDEX)
print('SPLIT_DIR:', SPLIT_DIR)
print('ANNOTATION_DIR:', ANNOTATION_DIR)
print('MODEL_DIR:', MODEL_DIR)

## Load the split spaCy corpus from Notebook 05a

Notebook 05a saved the annotated corpus as split `.spacy` files so that later notebooks do not need to rerun the full pipeline. Here we reload those documents and use them as the basis for concept annotation and training.

In [ ]:
spacy_files = sorted(SPLIT_DIR.glob('*.spacy'))
print(f'\nFound {len(spacy_files)} split files.')

nlp_base = spacy.load('en_core_web_sm', disable=['ner'])

docs = []
for fp in spacy_files:
    db = DocBin().from_disk(fp)
    docs.extend(list(db.get_docs(nlp_base.vocab)))

print('Loaded docs:', len(docs))

---

## Build annotation templates

Students annotate concept spans using **token-level BIO labels**. We export sentence-level TSV templates that can be opened in spreadsheet software or plain-text editors.

Each row corresponds to one token and contains:
- `doc_id`
- `sentence_id`
- `token_id`
- `token`
- `tag` (default = `O`)

In [ ]:
def export_bio_template(doc, doc_id: int, out_dir: Path, max_sentences: int = 25):
    """
    Export a sentence‑level BIO annotation template for one document.

    - Skips tokens that are spaces (tok.is_space) to avoid blank lines.
    - Strips extra whitespace from token text.
    - Assigns token_id as the index among *non‑space* tokens in the sentence.
    - Saves as a TSV with columns: doc_id, chunk_index, sentence_id, token_id, token, tag.
    """
    rows = []

    for sent_i, sent in enumerate(doc.sents):
        if sent_i >= max_sentences:
            break

        token_id_within_sent = 0
        for tok in sent:
            if tok.is_space:          # skip spaces entirely
                continue
            rows.append({
                'doc_id': doc_id,
                'chunk_index': doc.user_data.get('chunk_index'),
                'sentence_id': sent_i,
                'token_id': token_id_within_sent,
                'token': tok.text.strip(),   # remove any accidental whitespace
                'tag': 'O',
            })
            token_id_within_sent += 1

    out = pd.DataFrame(rows)
    out_path = out_dir / f'annotation_doc_{doc_id}.tsv'
    out.to_csv(out_path, sep='\t', index=False)
    return out_path

In [ ]:
# Example usage
example_paths = []
for i, doc in enumerate(docs[:100]):                     # or your list of docs
    doc_id = doc.user_data.get('pg_id', i)               # fallback to index if missing
    fp = export_bio_template(doc, doc_id, ANNOTATION_DIR)
    example_paths.append(fp)

print('\nExported annotation templates:', len(example_paths))
# for fp in example_paths:
#     print(fp)

## Suggested annotation workflow

A practical classroom workflow is:

1. Each student annotates a primary subset of templates.
2. Each student additionally annotates one shared template from another student.
3. Students compare disagreements.
4. A proofreading/adjudication pass resolves obvious span and label inconsistencies.
5. Only validated files are used for training.

### Reflection question
What kinds of disagreement would you expect for philosophical concepts: span boundary disagreement, concept/non-concept disagreement, or disagreement about concept granularity?

## Inter-annotator agreement

Each student annotates a primary subset and additionally revise an already annotated subset from another student for comparison.

Agreement analysis matters because:
- concepts may be ambiguous
- span boundaries may differ
- philosophical terminology shifts historically
- disagreement reveals methodological uncertainty

For simplicity, we compute **token-level agreement** in this notebook.

# Fill the gap

### You can create your own dataframes and use the function `token_level_agreement`

In [ ]:
def token_level_agreement(df_a: pd.DataFrame, df_b: pd.DataFrame) -> float:
    """Compute simple token-level agreement between two BIO annotation tables."""
    merged = df_a.merge(
        df_b,
        on=['doc_id', 'chunk_index', 'sentence_id', 'token_id', 'token'],
        suffixes=('_a', '_b')
    )

    if len(merged) == 0:
        return np.nan

    return (merged['tag_a'] == merged['tag_b']).mean()

print('Example usage:')
print('agreement = token_level_agreement(df_student1, df_student2)')

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# Use the function above or the module below

### Alternatively, you can store the files that you want to compare in two separate folders and use the following python module

In [ ]:
from nb10_inter_annotator_agreement import compare_annotations
# =============================================== YOUR CODE HERE ===============================================
summary = compare_annotations(
              folder_a="./data/processed/annotationsA/", # Modify to fit your setting
              folder_b="./data/processed/annotationsB/", # Modify to fit your setting
              label_a="A",
              label_b="B",
              ext=".tsv"
)

## Validation checks

Before training, annotation files should be validated:

- BIO tags should be legal
- `I-CONCEPT` should not appear without a preceding `B-CONCEPT`
- empty tokens should not exist
- sentence boundaries should remain intact

These checks prevent subtle annotation errors from breaking training or producing misleading results.

In [ ]:
VALID_TAGS = {'O', 'B-CONCEPT', 'I-CONCEPT'}

def validate_bio(df: pd.DataFrame) -> list[str]:
    """Validate a token-level BIO annotation table and return a list of issues."""
    issues = []

    required = ['doc_id', 'sentence_id', 'token_id', 'token', 'tag']
    missing = [c for c in required if c not in df.columns]
    if missing:
        issues.append(f'Missing required columns: {missing}')
        return issues

    bad_tags = sorted(set(df['tag'].dropna()) - VALID_TAGS)
    if bad_tags:
        issues.append(f'Illegal tags found: {bad_tags}')

    if df['token'].astype(str).str.strip().eq('').any():
        issues.append('Empty token strings found.')

    for (_, sent_df) in df.groupby(['doc_id', 'sentence_id']):
        prev = 'O'
        for _, row in sent_df.sort_values('token_id').iterrows():
            tag = row['tag']
            if tag == 'I-CONCEPT' and prev not in {'B-CONCEPT', 'I-CONCEPT'}:
                issues.append(
                    f"I-CONCEPT without preceding concept span in doc {row['doc_id']}, sent {row['sentence_id']}, token {row['token_id']}"
                )
            prev = tag

    return issues

print('Example usage:')
print('issues = validate_bio(annotation_df)')

## Load completed annotation files

Once students have annotated the TSV templates, we reload them here. The code below expects TSV files inside `data/annotations/`.

You can keep multiple versions of the same document during classroom review, but only the final validated files should be used for training.

In [ ]:
annotation_files = sorted(ANNOTATION_DIR.glob('annotation_doc_*A.tsv'))
print('Annotation files found:', len(annotation_files))

if annotation_files:
    ann_example = pd.read_csv(annotation_files[0], sep='\t')
    display(ann_example.head())
else:
    print('No TSV annotation files found yet. Export templates above, annotate them, then rerun this section.')

In [ ]:
validation_rows = []

for fp in annotation_files:
    df_ann = pd.read_csv(fp, sep='\t')
    issues = validate_bio(df_ann)
    validation_rows.append({
        'file': fp.name,
        'n_rows': len(df_ann),
        'n_issues': len(issues),
        'issues': ' | '.join(issues[:5]) if issues else ''
    })

validation_df = pd.DataFrame(validation_rows)
if len(validation_df):
    display(validation_df.head(20))
    validation_df.to_csv(TABLES_DIR / 'nb10_annotation_validation_summary.csv', index=False)
else:
    print('No validation summary to display yet.')

## BIO to span conversion

spaCy NER training expects entity spans in **character offsets**, not BIO labels. The next step converts validated token-level BIO annotations into `(start_char, end_char, label)` spans.

This conversion step is where annotation errors often surface, especially if token text no longer matches the underlying document or if span boundaries are inconsistent.

In [ ]:
def sentence_token_rows(doc, doc_id: int, max_sentences: int | None = None):
    """Yield token rows aligned with the exported TSV structure."""
    for sent_i, sent in enumerate(doc.sents):
        if max_sentences is not None and sent_i >= max_sentences:
            break
        for tok_i, tok in enumerate(sent):
            yield {
                'doc_id': doc_id,
                'chunk_index': doc.user_data.get('chunk_index'),
                'sentence_id': sent_i,
                'token_id': tok_i,
                'token': tok.text,
                'start_char': tok.idx,
                'end_char': tok.idx + len(tok),
            }

def bio_table_to_spans(doc, ann_df: pd.DataFrame):
    """Convert a BIO annotation table for one document into spaCy-style entity spans."""
    align_df = pd.DataFrame(list(sentence_token_rows(doc, int(ann_df['doc_id'].iloc[0]))))
    merged = align_df.merge(
        ann_df[['doc_id', 'chunk_index', 'sentence_id','token_id','token','tag']],
        on=['doc_id', 'sentence_id', 'token_id', 'token'],
        how='left'
    )

    spans = []
    current_start = None
    current_end = None

    for _, row in merged.sort_values(['sentence_id','token_id']).iterrows():
        tag = row['tag']

        if tag == 'B-CONCEPT':
            if current_start is not None:
                spans.append((current_start, current_end, 'CONCEPT'))
            current_start = int(row['start_char'])
            current_end = int(row['end_char'])

        elif tag == 'I-CONCEPT' and current_start is not None:
            current_end = int(row['end_char'])

        else:
            if current_start is not None:
                spans.append((current_start, current_end, 'CONCEPT'))
                current_start = None
                current_end = None

    if current_start is not None:
        spans.append((current_start, current_end, 'CONCEPT'))

    return spans

print('Functions defined: sentence_token_rows, bio_table_to_spans')

## Convert validated annotations into training examples

We now convert each validated TSV file into a spaCy training example.

Annotation files identify a document by `doc_id`, which is the **Project Gutenberg id (`pg_id`)** stored in each `Doc`'s `user_data` — not its position in the `docs` list. The next few cells build a `(pg_id, chunk_index) → Doc` mapping (`chunk_map`) and use it to look up the correct `Doc` for each annotation file, rather than indexing into `docs` positionally. If a file still has validation issues, it should be reviewed before being included.


## Mapping test: pg_id → chunk → Doc

In [ ]:
# Build mapping from (pg_id, chunk_index) to Doc
chunk_map = {}
for doc in docs:
    pg_id = doc.user_data.get('pg_id')
    chunk_index = doc.user_data.get('chunk_index', 0)
    if pg_id is not None:
        chunk_map[(pg_id, chunk_index)] = doc

print(f"Built chunk_map with {len(chunk_map)} (pg_id, chunk_index) pairs.")

# Check a sample (adjust these to known values)
sample_pairs = [(10108, 0), (10112, 0), (1016, 0)]
print("\nSample mapping check:")
for pg_id, chunk_index in sample_pairs:
    doc = chunk_map.get((pg_id, chunk_index))
    if doc:
        print(f"  pg_id {pg_id}, chunk {chunk_index}: found doc with {len(doc)} tokens")
    else:
        print(f"  pg_id {pg_id}, chunk {chunk_index}: NOT found")

# Check annotation files – do they have chunk_index column?
annotation_files = list(ANNOTATION_DIR.glob('*A.tsv'))
print(f"\nFound {len(annotation_files)} annotation files.")

for fp in annotation_files:
    ann_df = pd.read_csv(fp, sep='\t')
    has_chunk = 'chunk_index' in ann_df.columns
    doc_id = int(ann_df['doc_id'].iloc[0])
    if has_chunk:
        chunk_index = int(ann_df['chunk_index'].iloc[0])
        key = (doc_id, chunk_index)
    else:
        # Fallback: assume first chunk
        key = (doc_id, 0)
        print(f"⚠️  {fp.name} does not have chunk_id. Assuming chunk 0.")
    
    if key in chunk_map:
        print(f"✅ {fp.name}: pg_id {doc_id}, chunk {key[1]} found in mapping.")
    else:
        print(f"❌ {fp.name}: pg_id {doc_id}, chunk {key[1]} NOT found in mapping.")

## Prepare training examples from annotated chunks

In [ ]:
# Ensure chunk_map exists; if not, rebuild it
try:
    chunk_map
except NameError:
    chunk_map = {}
    for doc in docs:
        pg_id = doc.user_data.get('pg_id')
        chunk_index = doc.user_data.get('chunk_index', 0)
        if pg_id is not None:
            chunk_map[(pg_id, chunk_index)] = doc
    print(f"Rebuilt chunk_map with {len(chunk_map)} entries.")

train_examples = []
span_rows = []

annotation_files = list(ANNOTATION_DIR.glob('*A.tsv'))

for fp in annotation_files:
    ann_df = pd.read_csv(fp, sep='\t')
    
    # Validate
    issues = validate_bio(ann_df)
    if issues:
        print(f'Skipping {fp.name} because of validation issues.')
        continue

    # Get doc_id (pg_id) and chunk_index (fallback to 0)
    doc_id = int(ann_df['doc_id'].iloc[0])
    if 'chunk_index' in ann_df.columns:
        chunk_index = int(ann_df['chunk_index'].iloc[0])
    else:
        chunk_index = 0
        print(f"Note: {fp.name} does not have chunk_index. Assuming chunk 0.")

    # Retrieve the correct chunk
    doc = chunk_map.get((doc_id, chunk_index))
    if doc is None:
        print(f'Skipping {fp.name}: (pg_id {doc_id}, chunk {chunk_index}) not found.')
        continue

    # Convert BIO to spans
    spans = bio_table_to_spans(doc, ann_df)
    train_examples.append((doc.text, {'entities': spans}))

    for start, end, label in spans:
        span_rows.append({
            'doc_id': doc_id,
            'chunk_index': chunk_index,
            'start': start,
            'end': end,
            'label': label,
            'text': doc.text[start:end],
        })

span_df = pd.DataFrame(span_rows)
print(f'Training examples prepared: {len(train_examples)}')

if len(span_df):
    display(span_df.head(20))
    span_df.to_csv(TABLES_DIR / 'nb10-concept_spans.csv', index=False)
    print(f"Saved spans to {TABLES_DIR / 'nb10-concept_spans.csv'}")
else:
    print("No spans extracted – check annotation files and validation.")

## Train/dev split

Even for a small classroom dataset, we should hold out a development set for evaluation. The split below is lightweight and deterministic.

### Reflection question
If the annotated dataset is small, what kinds of evaluation instability should we expect?

In [ ]:
RANDOM_STATE = 42
TRAIN_FRACTION = 0.8

rng = np.random.default_rng(RANDOM_STATE)
order = rng.permutation(len(train_examples)) if len(train_examples) else np.array([], dtype=int)

cut = int(len(order) * TRAIN_FRACTION)
train_idx = order[:cut]
dev_idx = order[cut:]

train_data = [train_examples[i] for i in train_idx]
dev_data = [train_examples[i] for i in dev_idx]

print('Train examples:', len(train_data))
print('Dev examples:', len(dev_data))

## Save spaCy training corpora

We serialize the train/dev examples to `.spacy` files so they can be reused later without repeating conversion from TSV annotations.

In [ ]:
def examples_to_docbin(examples, vocab) -> DocBin:
    db = DocBin(store_user_data=True)
    nlp_tmp = spacy.blank('en')
    for text, ann in examples:
        doc = nlp_tmp.make_doc(text)
        ents = []
        for start, end, label in ann['entities']:
            span = doc.char_span(start, end, label=label, alignment_mode='contract')
            if span is not None:
                ents.append(span)
        doc.ents = ents
        db.add(doc)
    return db

if len(train_data):
    examples_to_docbin(train_data, nlp_base.vocab).to_disk(TRAIN_PATH)
    print('Saved train corpus:', TRAIN_PATH)
if len(dev_data):
    examples_to_docbin(dev_data, nlp_base.vocab).to_disk(DEV_PATH)
    print('Saved dev corpus:', DEV_PATH)

## Lightweight spaCy NER training on CPU

The goal here is not state-of-the-art performance. The goal is to build a **complete, inspectable training pipeline** that can run on student laptops.

We train a small custom spaCy NER model with one label: `CONCEPT`.

### Important methodological point
A small classroom model can still be useful even if it is imperfect, because the notebook is also about:
- what annotation decisions matter
- what the model gets wrong
- whether concept extraction is stable enough to support downstream historical claims


## Understanding `train_lightweight_ner`

This function trains a **custom Named Entity Recognition (NER) model** to detect a single label: `CONCEPT`. It does not start from a pre-trained spaCy model (like `en_core_web_sm`), but from a **blank English pipeline**. This is deliberate: we want the model to learn only what we teach it, without importing biases from modern news text.

### What happens inside the function

1. **Create a blank model**  
   `nlp = spacy.blank('en')`  
   This creates an empty pipeline with no components. A blank model knows nothing about language yet — no tokenizer, no POS tagger, no NER.

2. **Add a NER component**  
   `ner = nlp.add_pipe('ner')`  
   This adds an entity recogniser to the pipeline.

3. **Define the label**  
   `ner.add_label('CONCEPT')`  
   The model will learn to recognise only this one entity type. Any other span will be treated as “not an entity”.

4. **Initialise the model and optimiser**  
   `optimizer = nlp.initialize()`  
   This sets up the internal weights and prepares the optimiser that will adjust them during training.

5. **Training loop over epochs**  
   The function runs `n_iter` epochs (default 12). An **epoch** is one full pass through all training data.

6. **Shuffle the data**  
   `rng = np.random.default_rng(epoch + 1)`  
   `shuffled = [train_data[i] for i in rng.permutation(len(train_data))]`  
   The training examples are shuffled at the start of every epoch. This prevents the model from learning the order of examples, which would harm generalisation.

7. **Create mini-batches**  
   `for batch in minibatch(shuffled, size=8):`  
   Data is split into small batches of 8 examples. Mini-batching speeds up training and stabilises the weight updates.

8. **Convert annotations to spaCy’s format**  
   `examples.append(Example.from_dict(doc, ann))`  
   Each `(text, annotation)` pair is converted into an `Example` object. The annotation dictionary contains the character offsets of `CONCEPT` spans in that text.

9. **Update the model**  
   `nlp.update(examples, sgd=optimizer, drop=dropout, losses=losses)`  
   The model looks at the batch, computes the error between its current predictions and the true annotations, and updates its weights. `dropout` randomly ignores some network units during training, which helps prevent overfitting.

10. **Record the loss**  
    `losses_history.append({'epoch': epoch + 1, 'ner_loss': losses.get('ner', np.nan)})`  
    The NER component’s loss for each epoch is saved. Lower loss generally means the model is making fewer errors on the training data.

11. **Return the trained model and loss history**  
    The function returns the trained `nlp` object and a DataFrame with one row per epoch.

### What to observe while training

- The printed loss should generally **decrease** over epochs. If it oscillates wildly, you may need more data, fewer epochs, or different dropout.
- A very low loss does **not** guarantee good performance on new text. That is why we keep a separate test set and evaluate with precision, recall, and F1 afterwards.

### Important note about `dev_data`

The function currently accepts a `dev_data` argument but **does not use it** during training. You can use it after training to check performance on a held-out set and choose the best number of epochs, but the current version does not do early stopping or evaluation inside the loop.

This function is deliberately simple: its purpose is to make the training workflow transparent, not to produce a state-of-the-art model. By the end, you will have a small but meaningful NER system trained on your own annotations.

In [ ]:
from tqdm.auto import tqdm 

def train_lightweight_ner(train_data, dev_data=None, n_iter: int = 12, dropout: float = 0.25):
    """Train a lightweight spaCy NER model for the CONCEPT label."""
    nlp = spacy.blank('en')
    ner = nlp.add_pipe('ner')
    ner.add_label('CONCEPT')

    optimizer = nlp.initialize()
    losses_history = []

    # Wrap the epoch loop with tqdm for a progress bar
    for epoch in tqdm(range(n_iter), desc="Training NER", unit="epoch"):
        losses = {}
        rng = np.random.default_rng(epoch + 1)
        shuffled = [train_data[i] for i in rng.permutation(len(train_data))]

        for batch in minibatch(shuffled, size=8):
            examples = []
            for text, ann in batch:
                doc = nlp.make_doc(text)
                examples.append(Example.from_dict(doc, ann))
            nlp.update(examples, sgd=optimizer, drop=dropout, losses=losses)

        losses_history.append({'epoch': epoch + 1, 'ner_loss': losses.get('ner', np.nan)})
        # Optional: keep the print if you want the numerical loss as well
        tqdm.write(f"Epoch {epoch + 1:02d} | ner loss = {losses.get('ner', np.nan):.4f}")

    return nlp, pd.DataFrame(losses_history)

In [ ]:
if len(train_data) >= 2:
    nlp_ner, losses_df = train_lightweight_ner(train_data, dev_data=dev_data)
    display(losses_df)
else:
    print('Not enough training examples yet. Annotate more documents before training.')

In [ ]:
if 'losses_df' in globals() and len(losses_df):
    plt.figure(figsize=(8, 4))
    sns.lineplot(data=losses_df, x='epoch', y='ner_loss', marker='o')
    plt.title('Training loss across epochs')
    plt.tight_layout()
    plt.show()

    losses_df.to_csv(TABLES_DIR / 'nb10-training_loss.csv', index=False)

## Evaluate the model on the development set

We evaluate the model in two ways:

1. **simple token-level classification** (using BIO labels)
2. **qualitative inspection of predicted concept spans**

This is not a full benchmark, but it is enough for a classroom setting and highlights what the model is actually doing.

In [ ]:
def spans_to_token_bio(doc, spans):
    tags = ['O'] * len(doc)
    for start, end, label in spans:
        span = doc.char_span(start, end, label=label, alignment_mode='contract')
        if span is None:
            continue
        for i, tok in enumerate(span):
            tags[tok.i] = 'B-CONCEPT' if i == 0 else 'I-CONCEPT'
    return tags

def evaluate_token_level(nlp, dev_data):
    y_true, y_pred = [], []
    rows = []

    for text, ann in dev_data:
        gold_doc = nlp.make_doc(text)
        pred_doc = nlp(text)

        gold_tags = spans_to_token_bio(gold_doc, ann['entities'])
        pred_spans = [(ent.start_char, ent.end_char, ent.label_) for ent in pred_doc.ents]
        pred_tags = spans_to_token_bio(pred_doc, pred_spans)

        for tok_gold, tok_pred, tok in zip(gold_tags, pred_tags, gold_doc):
            y_true.append(tok_gold)
            y_pred.append(tok_pred)
            rows.append({
                'token': tok.text,
                'gold': tok_gold,
                'pred': tok_pred,
            })

    report = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
    return pd.DataFrame(report).T, pd.DataFrame(rows)

if 'nlp_ner' in globals() and len(dev_data):
    report_df, token_eval_df = evaluate_token_level(nlp_ner, dev_data)
    display(report_df)
    report_df.to_csv(TABLES_DIR / 'nb10-token_level_report.csv')
    token_eval_df.to_csv(TABLES_DIR / 'nb10-token_level_predictions.csv', index=False)
else:
    print('Train a model and prepare a dev set before evaluation.')

## Qualitative error analysis

Quantitative scores are not enough. We also want to inspect:

- false positives: where the model over-predicts concepts
- false negatives: where the model misses a concept
- boundary errors: concept detected, but span too short or too long
- ambiguity cases: prediction is plausible, but the gold label is debatable

### Reflection question
When the model disagrees with the annotation, is the model wrong, the annotation wrong, or is the concept itself genuinely ambiguous?

In [ ]:
def compare_gold_pred_examples(nlp, dev_data, n_examples: int = 8):
    rows = []
    for i, (text, ann) in enumerate(dev_data[:n_examples]):
        pred_doc = nlp(text)
        gold_spans = [(s, e, l) for s, e, l in ann['entities']]
        pred_spans = [(ent.start_char, ent.end_char, ent.label_) for ent in pred_doc.ents]

        rows.append({
            'example_id': i,
            'text_snippet': re.sub(r'\s+', ' ', text)[:250],
            'gold_spans': gold_spans,
            'pred_spans': pred_spans,
        })
    return pd.DataFrame(rows)

if 'nlp_ner' in globals() and len(dev_data):
    qual_df = compare_gold_pred_examples(nlp_ner, dev_data)
    display(qual_df)
    qual_df.to_csv(TABLES_DIR / 'nb10-qualitative_error_analysis.csv', index=False)

## Common failure modes

When you inspect the model, pay attention to these common failure modes:

- **boundary drift**: the model tags part of a concept phrase but not the full span
- **lexical memorization**: the model overfits recurring phrases in the annotated sample
- **context blindness**: the same word is not always a concept in every context
- **historical instability**: the meaning and role of a concept shifts over time
- **annotation sparsity**: too few positive examples to generalize reliably

A good notebook report should not just say "the score is low/high". It should explain **why** the model succeeds or fails.

## Save the trained model

If training completed successfully, we save the pipeline so it can be reused for inference in later experiments or compared with a future GPU-trained model.

In [ ]:
if 'nlp_ner' in globals():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    nlp_ner.to_disk(MODEL_DIR)
    print('Saved model to:', MODEL_DIR)

In [ ]:
# ------------------------------------------------------------
# Optional: use the trained model to auto-annotate texts in BIO TSV format
# (with chunking for long texts, and optional document limit)
# ------------------------------------------------------------
# This cell:
# 1) loads the saved spaCy NER model
# 2) adds a sentencizer if missing
# 3) reads cleaned text files and splits them into chunks (if needed)
# 4) predicts CONCEPT spans on each chunk
# 5) converts predictions to token‑level BIO tags
# 6) merges chunks into one TSV per text with filename prefix "AUTO_"
#
# Output columns:
# - doc_id
# - sentence_id   (continuous across chunks)
# - token_id      (per sentence)
# - token
# - tag

# --- CONFIGURATION ---
MAX_DOCS = 10         # set to None for all documents, or to a number (e.g., 10) for testing
CHUNK_SIZE = 200000   # characters per chunk

MODEL_DIR = Path("analysis/models/nb10-concept_ner")
TEXTS_DIR = Path("data/processed/cleaned")
DOC_INDEX = Path("analysis/tables/nb03-doc_index.csv")
AUTO_DIR = Path("data/processed/annotations/auto")
AUTO_DIR.mkdir(parents=True, exist_ok=True)

# --- load model ---
nlp_ner = spacy.load(MODEL_DIR)
nlp_ner.max_length = 2000000

if "sentencizer" not in nlp_ner.pipe_names:
    nlp_ner.add_pipe("sentencizer", before="ner")
    print("Added sentencizer to the pipeline.")

print("Loaded model from:", MODEL_DIR)
print("Pipeline components:", nlp_ner.pipe_names)

# --- load document index ---
doc_index = pd.read_csv(DOC_INDEX)

# --- apply document limit ---
if MAX_DOCS is not None and MAX_DOCS > 0:
    doc_index = doc_index.head(MAX_DOCS).copy()
    print(f"Annotating only first {MAX_DOCS} documents.")
else:
    print("Annotating all documents.")

# --- chunking function ---
def chunk_text(text, max_chars=200000):
    if len(text) <= max_chars:
        return [text]
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = min(start + max_chars, text_len)
        if end < text_len:
            break_pos = text.rfind('\n', start, end)
            if break_pos == -1:
                break_pos = text.rfind(' ', start, end)
            if break_pos == -1 or break_pos <= start:
                break_pos = end
            chunks.append(text[start:break_pos])
            start = break_pos
            while start < text_len and text[start].isspace():
                start += 1
        else:
            chunks.append(text[start:end])
            start = end
    return chunks

# --- helper: convert a Doc to BIO rows with sentence offset ---
def doc_to_bio_rows_with_offset(doc, doc_id, sent_offset):
    rows = []
    for sent_i, sent in enumerate(doc.sents):
        for tok_i, tok in enumerate(sent):
            tag = "O"
            for ent in doc.ents:
                if ent.label_ == "CONCEPT" and tok.i >= ent.start and tok.i < ent.end:
                    if tok.i == ent.start:
                        tag = "B-CONCEPT"
                    else:
                        tag = "I-CONCEPT"
                    break
            rows.append({
                "doc_id": doc_id,
                "sentence_id": sent_i + sent_offset,
                "token_id": tok_i,
                "token": tok.text,
                "tag": tag,
            })
    return rows

# --- annotate texts and save TSV files ---
saved = []

for i, row in tqdm(doc_index.iterrows(), total=len(doc_index), desc="Annotating texts"):
    filename = f"pg{int(doc_index['pg_id'].iloc[i])}.txt"
    text_path = TEXTS_DIR / filename

    if not text_path.exists():
        print(f"Skipping missing file: {filename}")
        continue

    text = text_path.read_text(encoding="utf-8", errors="replace")
    chunks = chunk_text(text, max_chars=CHUNK_SIZE)
    all_rows = []
    sent_offset = 0

    for chunk in chunks:
        doc = nlp_ner(chunk)
        rows = doc_to_bio_rows_with_offset(doc, doc_id=i, sent_offset=sent_offset)
        all_rows.extend(rows)
        sent_offset += len(list(doc.sents))

    out_df = pd.DataFrame(all_rows)
    out_name = f"AUTO_{Path(filename).stem}.tsv"
    out_path = AUTO_DIR / out_name
    out_df.to_csv(out_path, sep="\t", index=False)
    saved.append(out_path.name)

print(f"Saved {len(saved)} auto-annotated TSV files to {AUTO_DIR}")
print("Examples:", saved[:5])

## Reporting standards

When you report results from this notebook, always specify:

- what exactly was annotated (`CONCEPT`)
- the span policy used
- how many documents/sentences/tokens were annotated
- whether dates and periods are historically mixed
- how train/dev data were split
- what evaluation metric is being shown
- what common failure modes you observed
- whether outputs are being interpreted as exploratory or as robust evidence

### Final reflection
Can a concept-extraction model support claims about knowledge dynamics over time? Under what conditions would those claims be convincing, and when would they be too strong?

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A10 highlight;
```